In [ ]:
# ============================================================
# 1. Environment + evaluation configuration
# ============================================================

import os, gc, csv, json, math, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16

MODEL_NAME = "Qwen/Qwen2.5-1.5B"
MAX_LENGTH = 2048
NUM_SHOT = 5
MMLU_EVAL_BATCH_SIZE = 4

# ----------------------------------------------------------------
# Existing Phase-1 FULL-9000 checkpoints
# Set this to the directory containing epoch_1 ... epoch_4.
# ----------------------------------------------------------------
FULL_9000_CHECKPOINT_ROOT = Path(
    "/kaggle/input/notebooks/tanmairaghava/phase1-lora-adapting/checkpoints"
)

# ----------------------------------------------------------------
# New Phase-4 checkpoints from the training notebook.
# Change this if your Kaggle output/input path differs.
# ----------------------------------------------------------------
PHASE4_ROOT = Path("./less_phase4_training/experiments")

RANDOM450_CHECKPOINT_ROOT = PHASE4_ROOT / "random450" / "checkpoints"
LESS450_CHECKPOINT_ROOT = PHASE4_ROOT / "less450" / "checkpoints"

# MMLU location used in Phase 3.
MMLU_ROOT = Path("/kaggle/input/datasets/lokeswarareddyp/mmlu-data/mmlu")
MMLU_DEV_DIR = MMLU_ROOT / "dev"
MMLU_TEST_DIR = MMLU_ROOT / "test"

RESULTS_ROOT = Path("./less_phase5_mmlu_results")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Device :", DEVICE)
print("Model  :", MODEL_NAME)
print("MMLU   :", MMLU_ROOT)
print("Full checkpoints :", FULL_9000_CHECKPOINT_ROOT)
print("Random-450 checkpoints:", RANDOM450_CHECKPOINT_ROOT)
print("LESS-450 checkpoints  :", LESS450_CHECKPOINT_ROOT)


In [ ]:
# ============================================================
# 2. Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer:", MODEL_NAME)
print("Vocab:", len(tokenizer))


# LESS — Phase 5
## 5-shot MMLU evaluation

This notebook performs **only evaluation**.

It evaluates four conditions:

1. Base Qwen2.5-1.5B
2. Full-9000 using the **existing Phase-1 LoRA checkpoints**
3. Random-450 checkpoints produced by the Phase-4 training notebook
4. LESS-450 checkpoints produced by the Phase-4 training notebook

There is **no training** in this notebook.

MMLU:
- 57 subjects
- first 5 `dev` examples are the fixed demonstrations
- all `test` examples remain held out
- final score = mean accuracy across the 57 subjects


In [ ]:
# ============================================================
# 14. Load the 57 MMLU subjects and fixed 5-shot demonstrations
# ============================================================

dev_files = sorted(MMLU_DEV_DIR.glob("*_dev.csv"))
test_files = sorted(MMLU_TEST_DIR.glob("*_test.csv"))

dev_subjects = {
    p.name.removesuffix("_dev.csv")
    for p in dev_files
}
test_subjects = {
    p.name.removesuffix("_test.csv")
    for p in test_files
}

assert len(dev_subjects) == 57, f"Expected 57 dev subjects, found {len(dev_subjects)}"
assert len(test_subjects) == 57, f"Expected 57 test subjects, found {len(test_subjects)}"
assert dev_subjects == test_subjects, "Dev/test subject sets differ."

MMLU_SUBJECTS = sorted(dev_subjects)

def read_mmlu_csv(path: Path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)

        for row in reader:
            if not row:
                continue

            if len(row) < 6:
                raise ValueError(f"Malformed MMLU row in {path}: {row}")

            answer = row[5].strip()
            assert answer in ["A", "B", "C", "D"]

            rows.append({
                "question": row[0],
                "choices": row[1:5],
                "answer_letter": answer,
            })

    return rows


MMLU_DEV = {}
MMLU_TEST = {}

for subject in MMLU_SUBJECTS:
    dev_rows = read_mmlu_csv(
        MMLU_DEV_DIR / f"{subject}_dev.csv"
    )
    test_rows = read_mmlu_csv(
        MMLU_TEST_DIR / f"{subject}_test.csv"
    )

    assert len(dev_rows) >= NUM_SHOT

    MMLU_DEV[subject] = dev_rows
    MMLU_TEST[subject] = test_rows

MMLU_DEMOS = {
    subject: MMLU_DEV[subject][:NUM_SHOT]
    for subject in MMLU_SUBJECTS
}

print("Subjects :", len(MMLU_SUBJECTS))
print("5-shot demos:", sum(len(v) for v in MMLU_DEMOS.values()))

for subject in MMLU_SUBJECTS[:5]:
    print(subject, "->", len(MMLU_DEMOS[subject]), "demos,", len(MMLU_TEST[subject]), "test")


In [ ]:
# ============================================================
# 15. MMLU prompt construction
#     The content format mirrors Phase 3's MMLU messages:
#       user: question + A/B/C/D choices
#       assistant: answer letter
# ============================================================

ANSWER_LETTERS = ["A", "B", "C", "D"]

def mmlu_user_content(example):
    choices = example["choices"]

    return (
        f"{example['question']}\n\n"
        f"A. {choices[0]}\n"
        f"B. {choices[1]}\n"
        f"C. {choices[2]}\n"
        f"D. {choices[3]}"
    )

def build_few_shot_messages(demos, query):
    messages = []

    for demo in demos:
        messages.append({
            "role": "user",
            "content": mmlu_user_content(demo),
        })
        messages.append({
            "role": "assistant",
            "content": demo["answer_letter"],
        })

    messages.append({
        "role": "user",
        "content": mmlu_user_content(query),
    })

    return messages


# Verify the exact tokenizer IDs of A/B/C/D.
answer_token_ids = {}

for letter in ANSWER_LETTERS:
    ids = tokenizer.encode(
        letter,
        add_special_tokens=False,
    )

    if len(ids) != 1:
        raise RuntimeError(
            f"MMLU answer {letter!r} is not a single token for "
            f"{MODEL_NAME}: ids={ids}"
        )

    answer_token_ids[letter] = ids[0]

print("Answer token IDs:", answer_token_ids)


In [ ]:
# ============================================================
# 16. Model loading for evaluation
# ============================================================

def load_eval_base():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=DTYPE,
        trust_remote_code=True,
    )

    model.config.use_cache = True
    model.to(DEVICE)
    model.eval()

    return model


def load_eval_checkpoint(checkpoint_dir):
    base_model = load_eval_base()

    model = PeftModel.from_pretrained(
        base_model,
        checkpoint_dir,
        is_trainable=False,
    )

    model.eval()
    return model


def cleanup_eval_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# 17. Batched MMLU evaluation
# ============================================================

@torch.inference_mode()
def evaluate_mmlu_model(
    model,
    model_name,
    batch_size=MMLU_EVAL_BATCH_SIZE,
):
    model.eval()

    subject_results = []

    for subject in tqdm(
        MMLU_SUBJECTS,
        desc=f"MMLU | {model_name}",
    ):
        demos = MMLU_DEMOS[subject]
        test_rows = MMLU_TEST[subject]

        correct = 0
        total = len(test_rows)

        prompts = [
            tokenizer.apply_chat_template(
                build_few_shot_messages(demos, row),
                tokenize=False,
                add_generation_prompt=True,
            )
            for row in test_rows
        ]

        gold = [row["answer_letter"] for row in test_rows]

        for start in range(0, total, batch_size):
            batch_prompts = prompts[start:start + batch_size]

            encoded = tokenizer(
                batch_prompts,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
                add_special_tokens=False,
            )

            input_ids = encoded["input_ids"].to(
                DEVICE,
                non_blocking=True,
            )
            attention_mask = encoded["attention_mask"].to(
                DEVICE,
                non_blocking=True,
            )

            with torch.autocast(
                device_type="cuda",
                dtype=DTYPE,
            ):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    use_cache=True,
                    return_dict=True,
                )

            # Right-padding: gather the logits at the final actual token.
            last_positions = attention_mask.sum(dim=1) - 1
            row_ids = torch.arange(
                input_ids.shape[0],
                device=DEVICE,
            )

            next_token_logits = outputs.logits[
                row_ids,
                last_positions,
            ]

            candidate_logits = torch.stack([
                next_token_logits[:, answer_token_ids["A"]],
                next_token_logits[:, answer_token_ids["B"]],
                next_token_logits[:, answer_token_ids["C"]],
                next_token_logits[:, answer_token_ids["D"]],
            ], dim=1)

            predictions = candidate_logits.argmax(
                dim=1
            ).detach().cpu().tolist()

            for pred_idx, pred in enumerate(predictions):
                predicted_letter = ANSWER_LETTERS[pred]
                if predicted_letter == gold[start + pred_idx]:
                    correct += 1

            del encoded, input_ids, attention_mask
            del outputs, last_positions, row_ids
            del next_token_logits, candidate_logits

        accuracy = correct / total

        subject_results.append({
            "subject": subject,
            "correct": correct,
            "total": total,
            "accuracy": accuracy,
        })

    mmlu_score = float(
        np.mean([
            x["accuracy"]
            for x in subject_results
        ])
    )

    result = {
        "model": model_name,
        "mmlu_5shot": mmlu_score,
        "num_subjects": len(subject_results),
        "subjects": subject_results,
    }

    print()
    print("=" * 80)
    print(f"MMLU RESULT: {model_name}")
    print("=" * 80)
    print(f"5-shot MMLU: {100 * mmlu_score:.4f}%")

    return result


In [ ]:
# ============================================================
# 18. Evaluate the BASE model
# ============================================================

all_eval_results = []

if RUN_BASE:
    base_model = load_eval_base()

    base_result = evaluate_mmlu_model(
        base_model,
        "base",
    )

    all_eval_results.append(base_result)

    with open(
        RESULTS_ROOT / "mmlu_base.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(base_result, f, indent=2)

    cleanup_eval_model(base_model)

    print("Base evaluation saved.")


## Restartable evaluation

The Base model has already been evaluated:

**Base MMLU 5-shot = 46.72055329996985%**

It is intentionally skipped.

For Full-9000, Random-450, and LESS-450, every completed epoch is saved
immediately as a JSON file. If a Kaggle session stops, rerun the evaluation
cell; existing epoch results are detected and skipped automatically.

In [ ]:
# ============================================================
# 7. RESTARTABLE MMLU EVALUATION
# ============================================================
#
# BASE IS ALREADY EVALUATED:
#   MMLU 5-shot = 46.72055329996985%
#
# It is intentionally NOT recomputed.
#
# Every completed epoch is saved immediately to its own JSON.
# If the Kaggle session stops, simply rerun this cell: completed
# epochs are loaded and skipped automatically.
# ============================================================

all_eval_results = []

def result_path_for(experiment_name, epoch):
    return RESULTS_ROOT / f"mmlu_{experiment_name}_epoch{epoch}.json"

def evaluate_checkpoint_series_restartable(checkpoint_root, experiment_name):
    for epoch in range(1, 5):
        result_path = result_path_for(experiment_name, epoch)

        # Resume: already completed.
        if result_path.exists():
            print()
            print("=" * 80)
            print(f"SKIPPING {experiment_name} | EPOCH {epoch}")
            print(f"Existing result: {result_path}")
            print("=" * 80)

            with open(result_path, "r", encoding="utf-8") as f:
                result = json.load(f)

            all_eval_results.append(result)
            print(f"Recovered MMLU: {100.0 * result['mmlu_5shot']:.4f}%")
            continue

        checkpoint_dir = checkpoint_root / f"epoch_{epoch}"

        if not checkpoint_dir.exists():
            raise FileNotFoundError(
                f"Missing {experiment_name} checkpoint: {checkpoint_dir}"
            )

        print()
        print("=" * 80)
        print(f"EVALUATING {experiment_name} | EPOCH {epoch}")
        print("=" * 80)
        print(f"Checkpoint: {checkpoint_dir}")

        model = None

        try:
            model = load_eval_checkpoint(checkpoint_dir)

            result = evaluate_mmlu_model(
                model,
                f"{experiment_name}_epoch{epoch}",
            )

            result["experiment"] = experiment_name
            result["epoch"] = epoch
            result["checkpoint"] = str(checkpoint_dir)

            # SAVE IMMEDIATELY after this epoch.
            with open(result_path, "w", encoding="utf-8") as f:
                json.dump(result, f, indent=2)

            all_eval_results.append(result)

            print(f"Saved: {result_path}")
            print(
                f"MMLU {experiment_name} epoch {epoch}: "
                f"{100.0 * result['mmlu_5shot']:.4f}%"
            )

        finally:
            if model is not None:
                cleanup_eval_model(model)


# ============================================================
# BASE — SKIPPED
# ============================================================
#
# Already completed:
#   Base MMLU 5-shot = 46.72055329996985%
#
# DO NOT uncomment unless you intentionally want to spend
# several hours recomputing Base.
#
# base_model = load_eval_base()
# base_result = evaluate_mmlu_model(base_model, "base")
# with open(RESULTS_ROOT / "mmlu_base.json", "w") as f:
#     json.dump(base_result, f, indent=2)
# cleanup_eval_model(base_model)


# ============================================================
# FULL-9000 — EXISTING PHASE-1 CHECKPOINTS
# ============================================================

evaluate_checkpoint_series_restartable(
    FULL_9000_CHECKPOINT_ROOT,
    "full9000",
)


# ============================================================
# RANDOM-450 — NEW PHASE-4 CHECKPOINTS
# ============================================================

evaluate_checkpoint_series_restartable(
    RANDOM450_CHECKPOINT_ROOT,
    "random450",
)


# ============================================================
# LESS-450 — NEW PHASE-4 CHECKPOINTS
# ============================================================

evaluate_checkpoint_series_restartable(
    LESS450_CHECKPOINT_ROOT,
    "less450",
)


print()
print("=" * 80)
print("ALL AVAILABLE MMLU EVALUATIONS COMPLETE")
print("=" * 80)

print("Base: 46.72055329996985% (precomputed; skipped)")
for r in all_eval_results:
    print(
        f"{r.get('experiment', 'unknown')}"
        f" | epoch={r.get('epoch', '-')}"
        f" | {100.0 * r['mmlu_5shot']:.4f}%"
    )


In [ ]:
# ============================================================
# 8. FINAL SUMMARY
# ============================================================

BASE_MMLU = 0.4672055329996985

summary_rows = [{
    "model": "Qwen2.5-1.5B",
    "experiment": "base",
    "epoch": 0,
    "MMLU_5shot_percent": 100.0 * BASE_MMLU,
}]

for result in all_eval_results:
    summary_rows.append({
        "model": result.get("model", "Qwen/Qwen2.5-1.5B"),
        "experiment": result.get("experiment", "unknown"),
        "epoch": result.get("epoch", 0),
        "MMLU_5shot_percent": 100.0 * result["mmlu_5shot"],
    })

summary_df = pd.DataFrame(summary_rows)

order = {"base": 0, "full9000": 1, "random450": 2, "less450": 3}
summary_df["_order"] = summary_df["experiment"].map(order).fillna(99)

summary_df = (
    summary_df.sort_values(["_order", "epoch"])
    .drop(columns=["_order"])
    .reset_index(drop=True)
)

display(summary_df)

summary_csv = RESULTS_ROOT / "mmlu_summary.csv"
summary_json = RESULTS_ROOT / "mmlu_summary.json"

summary_df.to_csv(summary_csv, index=False)

with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summary_rows, f, indent=2)

print("Summary CSV :", summary_csv)
print("Summary JSON:", summary_json)


In [ ]:
# ============================================================
# 9. HEADLINE COMPARISON
# ============================================================

print("=" * 80)
print("MMLU 5-SHOT COMPARISON")
print("=" * 80)

print(f"Base       : {100.0 * BASE_MMLU:.4f}%")

for experiment in ["full9000", "random450", "less450"]:
    rows = summary_df[summary_df["experiment"].eq(experiment)].sort_values("epoch")

    if rows.empty:
        print(f"{experiment:11s}: no completed results yet")
        continue

    values = ", ".join(
        f"e{int(row.epoch)}={row.MMLU_5shot_percent:.4f}%"
        for row in rows.itertuples()
    )
    print(f"{experiment:11s}: {values}")
